# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

... подгружается по мере необходимости ..

# === ЭТАП 1 ===

# Загрузка первичных данных

Загружаем первичные данные из файлов:
- tracks.parquet
- catalog_names.parquet
- interactions.parquet

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

path = 'data/datasets/'
tracks = pd.read_parquet(path + 'tracks.parquet')
catalog = pd.read_parquet(path + 'catalog_names.parquet')
interactions = pd.read_parquet(path + 'interactions.parquet')

# Обзор данных

Проверяем данные, есть ли с ними явные проблемы.

### Данных о треках

In [ ]:
# общий обзор (и пустые значения)
display(tracks)
display(tracks.info())

In [ ]:
# проверка на дубликаты (необходимо преобразовать массивы в tuples для возможности хеширования)
pd.concat([
        tracks['track_id'], 
        tracks['albums'].apply(tuple),
        tracks['artists'].apply(tuple),
        tracks['genres'].apply(tuple)
    ], axis=1).duplicated().sum()

### Данных каталога

In [ ]:
# общий обзор (и пустые значения)
display(catalog)
display(catalog.info(show_counts=True))

In [ ]:
# проверка на дубликаты
catalog.duplicated().sum()

In [ ]:
catalog.type.unique()

In [ ]:
display('album', catalog.query('type == "album"'))
display('artist', catalog.query('type == "artist"'))
display('artist', catalog.query('type == "artist"'))
display('track', catalog.query('type == "track"'))

### Данных взаимодействий

In [ ]:
# общий обзор (и пустые значения)
display(interactions)
display(interactions.info(show_counts=True))

In [ ]:
# почему индекс всего до 291 ?
display(interactions.index[:30])
display(interactions.head(30))

# понятно - сборная куча историй по каждому пользователю, без переиндексации финального датасета

In [ ]:
# проверим за какой диапазон дат есть история, и нет ли выбросов

interactions.started_at.min(), interactions.started_at.max()

### Проверка - есть ли треки с неизвестными исполнителями, альбомами, жанрами

#### Проверяем сначала треки, у которых не указаны связанные сущности

Треки, у которых нет альбомов - проверим просто для интереса, но в целом тут нет какой-то проблемы, что трек не добавлен в альбом

In [ ]:
# проверка записей, у которых не заданы связанные сущности

track_empty_albums = tracks[tracks['albums'].apply(len) == 0]
print(f'tracks with empty albums = {track_empty_albums.shape[0]}')

# 18 треков без альбомов - очень мало, значит почти каждый трек есть в каком-то альбоме .. ну ок

Треки, у которых нет исполнителя или жанра - это проблема или ошибка в данных - эти треки надо будет исключить

In [ ]:
# проверка записей, у которых не заданы связанные сущности

track_empty_arists = tracks[tracks['artists'].apply(len) == 0]
track_empty_genres = tracks[tracks['genres'].apply(len) == 0]

print(f'tracks with empty arists = {track_empty_arists.shape[0]}')
print(f'tracks with empty genres = {track_empty_genres.shape[0]}')
print(f'total number of track with empty items = {track_empty_arists.shape[0] + track_empty_genres.shape[0]}')

# Проблемные треки:
#   - tracks with empty arists = 15369
#   - tracks with empty genres = 3687
#   - total number of track with emptыy items = 19056

#### Проверяем треки, у которых установлены ссылки на объекты, которых не существуюет

In [42]:
# сформируем плоское представление записей, чтобы проверить корректность ссылок на связанные объекты
tracks_flat = tracks \
    .explode('albums', ignore_index=True) \
    .explode('artists', ignore_index=True) \
    .explode('genres', ignore_index=True) \
    .rename(columns={'albums': 'album_id', 'artists': 'artist_id', 'genres': 'genre_id'})

In [ ]:
# дополняем данными о связанных объектах (имеется только название из каталога)
fields = catalog.type.unique()
for f in fields:
    tracks_flat = tracks_flat.merge(
        catalog.query('type == @f').set_index('id')['name'].rename(f + '_name'),
        left_on=f+'_id',
        right_index=True,
        how='left'
    )
tracks_flat

Проверяем ошибки в ссылках - то есть id объекта есть в tracks, но название связанной сущности не было найдено в каталоге (после merge с catalog)

In [ ]:
# треки без названий

tracks_flat[tracks_flat['track_id'].notna() & tracks_flat['track_name'].isna()]

In [ ]:
# альбомы без названий

tracks_flat[tracks_flat['album_id'].notna() & tracks_flat['album_name'].isna()]

In [ ]:
# исполнители без названий

tracks_flat[tracks_flat['artist_id'].notna() & tracks_flat['artist_name'].isna()]

In [ ]:
# жанры без названий

tracks_unknown_genres = tracks_flat[tracks_flat['genre_id'].notna() & tracks_flat['genre_name'].isna()]

print(f'tracks with unknown genres = {tracks_unknown_genres["track_id"].nunique()}, unknown genres count: {len(tracks_unknown_genres["genre_id"].unique())}')
display(tracks_unknown_genres)
display('unknown genres ids:', tracks_unknown_genres['genre_id'].unique())

In [ ]:
# посмотрим на первую запись в исходных данные о треках - правда ли там ошибка в жанре
display(tracks.query('track_id == 436'))
display(catalog.query('id in [28, 164]'))

# - видим, что задана ссылка на жанр id = 164, но такого жанра в каталоге и правда нет

# Выводы

Приведём выводы по первому знакомству с данными:
- есть ли с данными явные проблемы,
- какие корректирующие действия (в целом) были предприняты.

### Результат обзора данных
- проблем с данными не обнаружено 
    - пропусков (na) и дублей строк нет
- типы данных
    - все идентификаторы целочисленные - ок
    - идентификаторы имеют пропуски, поэтому при работе с разреженными матрицами потребуется дополнительное кодирование (например, LabelEncoder)
- история взаимодействий
    - история представлена целиком за 2022г - от 01.01 до 31.12.2022, выбросов из этого диапазана нет
    - индекс в датасете не сквозной, а собран из историй по каждому пользователю - возможно потребуется переиндексировать в дальнейшем
- записи с незаполненными связанными объектами
    - с незаполненными альбомами = 18
    - с незаполненными исполнителями = 15369
    - с незаполненными жанрами = 3687
    - итого треков с незаполненными данными = 19074
- записи с некорректными связями (ссылками) на каталог
    - неизвестные треки - нет
    - неизвестные альбомы - нет
    - неизвестные исполнители - нет
    - неизвестные жарны - 48345
        - самих неизвестных жанров - 30
    - итого записей с неизвестными связями - 48345


### Корректировка в данных
- удалим данные о треках с незаполненными связанными исполнителями и жанрами
- удалим данные о треках с неизвестными связанными жанрами

In [ ]:
### clean tracks 
tracks_count_orig = tracks.shape[0]

tracks = tracks[~tracks.track_id.isin(
    pd.concat([
        track_empty_arists.track_id, 
        track_empty_genres.track_id, 
        tracks_unknown_genres.track_id
        ])
    )]

tracks_count_clean = tracks.shape[0]
print(f'tracks count original: {tracks_count_orig}, tracks count cleaned: {tracks_count_clean}, bad tracks removed: {tracks_count_orig - tracks_count_clean} ({100*((tracks_count_orig - tracks_count_clean)/tracks_count_orig):.2f} %)')

### clean interactions
interactions_count_orig = interactions.shape[0]

interactions = interactions[~interactions.track_id.isin(
    pd.concat([
        track_empty_arists.track_id, 
        track_empty_genres.track_id, 
        tracks_unknown_genres.track_id
        ])
    )]

interactions_count_clean = interactions.shape[0]
print(f'interactions count original: {interactions_count_orig}, interactions count cleaned: {interactions_count_clean}, bad interactions removed: {interactions_count_orig - interactions_count_clean} ({100*((interactions_count_orig - interactions_count_clean)/interactions_count_orig):.2f} %)')

In [ ]:
# кстати, треки с пустыми альбомами тоже ушли при очистке

tracks[tracks.track_id.isin(track_empty_albums.track_id)]

In [ ]:
# почистим также tracks_flat

tracks_flat.dropna(inplace=True, ignore_index=True)

tracks_flat = tracks_flat[~tracks_flat.track_id.isin(
    pd.concat([
        track_empty_arists.track_id, 
        track_empty_genres.track_id, 
        tracks_unknown_genres.track_id
        ])
    )]
print(tracks_flat.shape)

#### Итоговый датасет с треками

Сформируем итоговый датасет с треками, который содержит как идентификаторы, так и названия в одном представлении - для использования в дальнейших этапах

In [ ]:
tracks_flat_combined = tracks_flat.groupby(['track_id', 'track_name']).agg(
    {'album_id': 'unique',
    'artist_id': 'unique',
    'genre_id': 'unique',
    'album_name': 'unique',
    'artist_name': 'unique',
    'genre_name': 'unique'}).reset_index()
tracks_flat_combined

### Сохраним агрегированные данные о треках в файл, для быстрого извлечения при необходимости

In [56]:
data_path = 'data/preprocess/'

tracks.to_parquet(data_path + "tracks_clean.parquet")
interactions.to_parquet(data_path + "interactions_clean.parquet")

tracks_flat.to_parquet(data_path + "tracks_flat.parquet")
tracks_flat_combined.to_parquet(data_path + "tracks_flat_combined.parquet")

# === ЭТАП 2 ===

[выполнить kernel restart - для очистки памяти]

## Загрузим предобработанные данные
после шага 1, для удобства продолжения экспериментов

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

### initial
path = 'data/datasets/'
#tracks = pd.read_parquet(path + 'tracks.parquet')
catalog = pd.read_parquet(path + 'catalog_names.parquet')
#interactions = pd.read_parquet(path + 'interactions.parquet')

### preprocess
path = 'data/preprocess/'

#tracks = pd.read_parquet(path + 'tracks_clean.parquet')
interactions = pd.read_parquet(path + 'interactions_clean.parquet')

#tracks_flat = pd.read_parquet(path + "tracks_flat.parquet")
tracks_flat_combined = pd.read_parquet(path + "tracks_flat_combined.parquet")

### features
#path = 'data/preprocess/'
#genres = pd.read_parquet(path + "genres.parquet")

# EDA

## Распределение количества прослушанных треков.

#### Распределение по месяцам

In [ ]:
interactions.info()

In [ ]:
interactions_by_month = interactions.groupby(interactions['started_at'].dt.to_period('M')).agg(events=('started_at', 'count'), users=('user_id', 'nunique'), tracks=('track_id', 'nunique')).reset_index()

fig, ax = plt.subplots(3, 1, figsize=(10, 8))
interactions_by_month.plot('started_at', 'events', kind='bar', title='Всего прослушиваний vs месяцы', ax=ax[0])
interactions_by_month.plot('started_at', 'users', kind='bar', title='Уникальные пользователи vs месяцы', ax=ax[1])
interactions_by_month.plot('started_at', 'tracks', kind='bar', title='Уникальные треки vs месяцы', ax=ax[2])
fig.tight_layout()

interactions_by_month

### Распределение по пользователям
- сколько в среднем пользователь прослушал треков


In [ ]:
interactions_by_user = interactions[['track_id', 'user_id']].groupby('user_id').agg(track_count=('track_id', 'count')).sort_values(by='track_count', ascending=False).reset_index()
interactions_by_user

In [ ]:
# посмотрим квантили распределения 
interactions_by_user['track_count'].describe().astype(int)

# среднее = 156
# min = 1
# 25Q = 22
# медиана = 54
# 75Q = 149
# max = 16609

In [ ]:
import numpy as np 

ax = interactions_by_user['track_count'].plot.hist(loglog=True, bins=500, legend=True, title='Tracks by user histogram (Log scale)')
plt.vlines([22], colors='yellow', ymin=0, ymax=1e6, label='Q-25%', )
plt.text(22 + 1, 1, '22')
plt.vlines([54], colors='red', ymin=0, ymax=1e6, label='Q-50%')
plt.text(54 + 5, 1, '54')
plt.vlines([149], colors='cyan', ymin=0, ymax=1e6, label='Q-75%')
plt.text(149 + 5, 1, '149')

ax.legend()

### Выводы
- количество пользователей и треков возрастают со временем, сервис набирает популярность, в декабре наметился небольшой спад
- распределение количества прослушанных треков пользователем
    - среднее = 156
    - медиана = 54
    - мин = 1
    - макс = 16609
    - после Q75% = 149 треков количество пользователей логарифмически убывает, и количество треков логарифмически растет

## Распределение треков по жанрам
- рассмотрим на какие жанры распределяются треки

In [7]:
path = 'data/preprocess/'
tracks_flat = pd.read_parquet(path + "tracks_flat.parquet")

In [ ]:
tracks_flat[['track_id', 'genre_name']].drop_duplicates() \
    .groupby('genre_name')['track_id'].count().rename('count').sort_values(ascending=False) \
    .plot.bar(log=True, rot=90, figsize=(20,6), title='Tracks distribution by genres')

## Распределение количества жанров по трекам
- рассмотрим скольк жанров приходится на один трек

In [ ]:
genre_count = tracks_flat_combined['genre_id'].transform(len).rename('genre_count').value_counts().reset_index()
genre_count['ratio'] = genre_count['count'].transform(lambda x: x/genre_count['count'].sum())
genre_count

In [ ]:
tracks_flat_combined['genre_id'].transform(len).plot.hist(bins=100, log=True, xticks=range(11), title='Genres amount by track distribution (logscale)')

### Вывод
- у большинства треков 1 (57%) или 2 (40%) жанра - в сумме 97% всех треков
    - еще у 2% 3 жанра
    - и оставшийся 1% распределяется на 4-10 жанров
- так как жанров всего 131 то матрица распределения треков по жанрам будет заполнять 1-2 ячейки, то есть ультра разреженная

## Наиболее популярные треки

- популярность определяем по частности взаимодействия (поскольку данных о рейтингах нет)

In [ ]:
tracks_polularity = pd.DataFrame(interactions['track_id'].value_counts().sort_values(ascending=False)) \
    .merge(
        tracks_flat_combined.set_index('track_id'),
        on='track_id',
        how='left'
    ) \
    .reset_index()

tracks_polularity

In [ ]:
tracks_polularity[['count']].describe().astype(int)

### График топ-100 треков

In [ ]:
tracks_polularity[['track_name', 'count']] \
    .head(100) \
    .reset_index() \
    .set_index(['index', 'track_name']) \
    .plot.bar(legend=True, title='Tracks poularity', figsize=(20,5))

### Выводы
- популярность треков убывает линейным образом от лидеров к менее популярным
    - среднее - 230
    - мин - 5
    - медиана - 19
    - макс - 111062

## Наиболее популярные жанры

#### Общий список жанров

In [ ]:
# список жанров из каталога
genres = catalog.query('type == "genre"').reset_index(drop=True)
genres

In [52]:
# сохраним в файл на будущее
data_path = 'data/preprocess/'
genres.to_parquet(data_path + "genres.parquet")

#### Популярность жанров по истории взаимодействия
- будем определять популярность жанров, исходя из взаимодействий с треками - наиболее часто встречающиеся жанры по всем взаимодействиям со всеми треками (а не топ-треков)

In [ ]:
genres_popularity = pd.DataFrame(interactions['track_id'].value_counts().sort_values(ascending=False).reset_index()) \
    .merge(
        tracks_flat_combined.set_index('track_id')[['genre_id', 'genre_name']],
        on='track_id',
        how='left'
    ) \
    .explode(['genre_id', 'genre_name'])  \
    .groupby(['genre_id', 'genre_name']).agg(count=('count', 'sum')).sort_values(by='count', ascending=False) \
    .reset_index()       

genres_popularity

##### График популярности жанров

In [ ]:
genres_popularity.reset_index()[['index', 'genre_name', 'count']].set_index(['index', 'genre_name']) \
    .plot.bar(log=True, title='Genres popularity (logscale)', figsize=(20,5))

### Непопулярные жанры
Рассмотрим жанры, которые вообще не попали в активность - то есть "непопулярны"

In [ ]:
genres_not_popular = genres[~genres['id'].isin(genres_popularity['genre_id'])].reindex().reset_index(drop=True)
print(f'genres not popular (not in history) count = {genres_not_popular.shape[0]}')

genres_not_popular

In [ ]:
tracks_flat_combined[tracks_flat_combined['genre_id'].isin(genres_not_popular['id'])]

### Вывод 
- топ-3 жанра
    - pop
    - rap
    - allrock
- видно что есть 35 жанров, по которым нет истории взаимодействия
    - но и треков с этими жанрами нет

## Треки, которые никто не прослушал

In [ ]:
tracks_no_activity = tracks_flat_combined[~tracks_flat_combined['track_id'].isin(tracks_polularity['track_id'])]
tracks_no_activity

### Выводы
- треков, которые никто не прослушал, нет

# Преобразование данных

Преобразуем данные в формат, более пригодный для дальнейшего использования в расчётах рекомендаций.

In [ ]:
# tracks_flat_combined - содержит всю необходимую информацию в удобном виде
# переименуем track_id в item_id для унификации, и приведем к int32 для сокращения памяти

tracks_flat_combined.rename(columns={'track_id': 'item_id'}, inplace=True)
tracks_flat_combined['item_id'] = tracks_flat_combined['item_id'].astype('int32')
tracks_flat_combined

In [ ]:
tracks_flat_combined.info()

In [ ]:
# переименуем track_id в item_id для унификации

interactions.rename(columns={'track_id': 'item_id'}, inplace=True)
interactions

In [ ]:
interactions.info()

# Сохранение данных

Сохраним данные в двух файлах в персональном S3-бакете по пути `recsys/data/`:
- `items.parquet` — все данные о музыкальных треках,
- `events.parquet` — все данные о взаимодействиях.

In [7]:
# сначала сохраним локально 
path = 'data/recsys/'

tracks_flat_combined.sort_values(by='item_id', ignore_index=True).to_parquet(path + "items.parquet")
interactions.sort_values(by=['user_id','started_at'], ignore_index=True).to_parquet(path + "events.parquet")

In [1]:
# проверка файлов для отправки в S3

import pandas as pd

path = 'data/recsys/'
items = pd.read_parquet(path + "items.parquet")
events = pd.read_parquet(path + "events.parquet")

Сохранение в S3-хранилище

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# pip install s3fs

# теперь сохраним в S3

s3_storage = os.getenv('S3_ENDPOINT_URL')           # https://storage.yandexcloud.net
s3_bucket = os.getenv('S3_BUCKET_NAME')             # s3-student-mle-20260306-f5465b0629-freetrack/Sprint-4
s3_access_key = os.getenv('AWS_ACCESS_KEY_ID')      # YCAJE3Nlz8iDILW5VTYM1ihQB
s3_secret_key = os.getenv('AWS_SECRET_ACCESS_KEY')  # secret

path = "s3://" + s3_bucket + "/recsys/data/"
print(path)

# параметры подключения к S3 хранилищу
storage_options = {
    "client_kwargs": {"endpoint_url": s3_storage},
    "key": s3_access_key,  
    "secret": s3_secret_key,  
}

items.to_parquet(path + "items.parquet", storage_options=storage_options)
events.to_parquet(path + "events.parquet", storage_options=storage_options)

# Очистка памяти

Здесь, может понадобится очистка памяти для высвобождения ресурсов для выполнения кода ниже. 

Приведите соответствующие код, комментарии, например:
- код для удаление более ненужных переменных,
- комментарий, что следует перезапустить kernel, выполнить такие-то начальные секции и продолжить с этапа 3.

In [105]:
# 1 - промежуточные файлы сохранены в data/preprocess 
#   - так что для EDA-анализа по этапу 2 можно загрузить файлы в начале секции "Этапа 2" 
#   - и повторить любой из этапов без необходимости выполнения исходной последовательности

# 2 - все рабочие файлы для создания рекомендательной системы сохранены в 
#   - локально - data/recsys 
#   - в S3 - https://storage.yandexcloud.net/s3-student-mle-20260306-f5465b0629-freetrack/Sprint-4/recsys/data/

# - Этап 3 надо начинать с чистого листа и kerner restart

# === ЭТАП 3 ===

# Загрузка данных

Если необходимо, то загружаем items.parquet, events.parquet 
!!! для разных этапов может потребоваться раскоментировать конкретные строчки для их выполнения - это не статичный раздел

In [1]:
# проверка файлов для отправки в S3

import pandas as pd

path = 'data/recsys/'

items = pd.read_parquet(path + "items.parquet")
events = pd.read_parquet(path + "events.parquet")

#--- т.к. даже 32 Gb памяти на ВМ не хватает для полного прогона формирования персональных ALS рекомендаций, то потребуется начинать с промежутончых этапов

# промежуточные результаты, для выполнения разных этапов подготовки - нужно раскрыть нужные комментарии для точечных задач
#events_train = pd.read_parquet(path + 'events_train.parquet')


In [ ]:
items

In [ ]:
items.info()

In [ ]:
events

In [ ]:
events.info()

# Разбиение данных

Разбиваем данные на тренировочную, тестовую выборки.

In [ ]:
train_date_split = '2022-12-16'
events_train = events.query('started_at < @train_date_split').copy()
events_test = events.query('started_at >= @train_date_split').copy()

print(f'events_train size: {events_train.shape[0]:,} ({(100*events_train.shape[0]/events.shape[0]):.2f}%), events_test size: {events_test.shape[0]:,} ({(100*events_test.shape[0]/events.shape[0]):.2f}%)')

# Топ популярных

Рассчитаем рекомендации как топ популярных, рассчитанных на этапе 2 EDA - "Наиболее популярные треки" и сохраненные в соответсвующий файл

In [ ]:
tracks_polularity = pd.DataFrame(events_train[['item_id']].value_counts().sort_values(ascending=False))
tracks_polularity

## Сохраним топ-100 треков в top_popular.parquet

In [11]:
path = 'data/recsys/'
top_n = 100
tracks_polularity.head(top_n).to_parquet(path + 'top_popular.parquet')

# Персональные

Рассчитаем персональные рекомендации.

## Закодируем идентификторы объектов для подготовки csr-матрицы

In [ ]:
from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
user_encoder.fit(events.user_id)

item_encoder = LabelEncoder()
item_encoder.fit(events.item_id)

Сохраним кодировщики в файл, для будущего переиспользования и экономии памяти на последующих этапах

In [ ]:
import joblib

path = 'data/recsys/'
joblib.dump(user_encoder, path + 'user_encoder.joblib')
joblib.dump(item_encoder, path + 'item_encoder.joblib')

Загрузка из файлов при необходимости на нужном этапе

In [3]:
import joblib

path = 'data/recsys/'
user_encoder = joblib.load(path + 'user_encoder.joblib')
item_encoder = joblib.load(path + 'item_encoder.joblib')

In [4]:
events_train['user_id_enc'] = user_encoder.transform(events_train.user_id)
events_train['item_id_enc'] = item_encoder.transform(events_train.item_id)

# сократим потребление памяти с int64 на int32
events_train = events_train.astype({'user_id_enc': 'int32', 'item_id_enc': 'int32'})

Выполним промежуточное сохранение events_train с закодированными идентификаторами в файл

In [5]:
path = 'data/recsys/'
events_train.to_parquet(path + 'events_train.parquet')

Подготовим также данные для events_test

In [6]:
events_test['user_id_enc'] = user_encoder.transform(events_test.user_id)
events_test['item_id_enc'] = item_encoder.transform(events_test.item_id)
events_test = events_test.astype({'user_id_enc': 'int32', 'item_id_enc': 'int32'})

Выполним промежуточное сохранение events_test с закодированными идентификаторами в файл

In [7]:
path = 'data/recsys/'
events_test.to_parquet(path + 'events_test.parquet')

In [10]:
# пока не нужны, можно почистить
del events
del events_test

## Сформируем матрицу взаимодействия user-2-item

In [ ]:
from scipy.sparse import csr_matrix
import numpy as np

user2item_matrix_train = csr_matrix(
    (np.ones(events_train.shape[0], dtype=np.int8),
    (events_train.user_id_enc.values, events_train.item_id_enc.values))
)
user2item_matrix_train

## Обучим ALS-модель

In [ ]:
%env OPENBLAS_NUM_THREADS=1

from implicit.als import AlternatingLeastSquares

n_factors = 50
n_iterations = 15
model_als = AlternatingLeastSquares(factors=n_factors, iterations=n_iterations, regularization=0.05, random_state=15)
model_als.fit(user2item_matrix_train)

### Сохраним обученную модель в файл
- сэкономим 12 минут при повторных операциях

In [13]:
path = 'data/recsys/'
model_als.save(path + 'model_als_trained.npz')

### Загрузим обученную модель из файла 
- точнее обученные параметры

In [14]:
path = 'data/recsys/'

# this not works - got empty factors after load()
#   n_factors = 50
#   n_iterations = 15
#   model_als = AlternatingLeastSquares(factors=n_factors, iterations=n_iterations, regularization=0.05, random_state=15)
#   model_als.load(path + 'model_als_trained.npz')

# this works
model_als = AlternatingLeastSquares().load(path + 'model_als_trained.npz')


## Сформируем персональные топ-100 рекомендации для всех пользователей

In [ ]:
# clear as much memory as we can

#del events
#del events_train

In [17]:
# получаем список всех возможных user_id (перекодированных)
user_ids_encoded = range(len(user_encoder.classes_))

# получаем рекомендации для всех пользователей
als_recommendations_raw = model_als.recommend(
    user_ids_encoded, 
    user2item_matrix_train[user_ids_encoded], 
    filter_already_liked_items=False, 
    N=100)

item_ids_enc = als_recommendations_raw[0]
scores_als = als_recommendations_raw[1]

### Сохраним в файл сырые данные рекомендаций
- 46 минут генерации

In [18]:
import numpy as np

path = 'data/recsys/'
np.savez_compressed(path + 'als_recommendations_raw.npz', item_ids=item_ids_enc, scores=scores_als)

### Загрузка готовых сырых рекомендаций из файла

In [19]:
import numpy as np

path = 'data/recsys/'

loaded_recs = np.load(path + 'als_recommendations_raw.npz')
item_ids_enc = loaded_recs['item_ids']
scores_als = loaded_recs['scores']

## Сформируем рекомендации в удобном табличном формате

In [ ]:
import pandas as pd

# преобразуем полученные рекомендации в табличный формат (~15Gb memory required)

user_ids_encoded = range(len(user_encoder.classes_))

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(), 
    "score": scores_als.tolist()})
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")

# получаем изначальные идентификаторы
als_recommendations["user_id"] = user_encoder.inverse_transform(als_recommendations["user_id_enc"])
als_recommendations["item_id"] = item_encoder.inverse_transform(als_recommendations["item_id_enc"])
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"])

als_recommendations

## Сохраним полученные рекомендации в файл

In [21]:
als_recommendations = als_recommendations[["user_id", "item_id", "score"]]

path = 'data/recsys/'
als_recommendations.to_parquet(path + "personal_als.parquet")

можно выполнить kernel restart

## Проверим визуально как формируются персональные рекомендации

In [1]:
import pandas as pd

path = 'data/recsys/'
als_recommendations = pd.read_parquet(path + "personal_als.parquet")
events_train = pd.read_parquet(path + "events_train.parquet")
items = pd.read_parquet(path + "items.parquet")

Инициализировать кодировщики вручную из соответсвующей секции, чтобы не дублировать сюда

In [ ]:
import numpy as np

uid_enc = np.random.randint(0, 1000000)
uid = user_encoder.inverse_transform([uid_enc])[0]

print(f'pesonal history for user id = {uid} (10 recent tracks):')
user_history = events_train[events_train['user_id'] == uid] \
    .merge(items, on='item_id')

display(user_history.tail(10))

personal_recs = als_recommendations.query('user_id == @uid') \
    .merge(items, on='item_id')

print('pesonal recomendations (top 10):')
display(personal_recs.head(10))

трудно сказать .. но вроде есть схожесть 

# Похожие

Рассчитаем похожие, они позже пригодятся для онлайн-рекомендаций.

## Используем обученную сохраненную ALS-модель для получения похожих объектов

Загрузка модели

In [ ]:
from implicit.als import AlternatingLeastSquares

path = 'data/recsys/'
model_als = AlternatingLeastSquares().load(path + 'model_als_trained.npz')

## Получение 10 похожих объектов для всех объектов

In [7]:
item_ids_encoded = range(len(item_encoder.classes_))

sim_item_ids_enc, sim_scores_als = model_als.similar_items(itemid=item_ids_encoded, N=10)

Сохраним сырые результаты в файл, чтобы не обучать постоянно повторно (> 41 min)

In [8]:
path = 'data/recsys/'
np.savez_compressed(path + 'sim_als_recommendations_raw.npz', item_ids=sim_item_ids_enc, scores=sim_scores_als)

Восстановление из файла

In [9]:
import numpy as np

path = 'data/recsys/'

loaded_recs = np.load(path + 'sim_als_recommendations_raw.npz')
sim_item_ids_enc = loaded_recs['item_ids']
sim_scores_als = loaded_recs['scores']

## Приведем к удобной табличной форме

In [ ]:
import pandas as pd

# преобразуем полученные рекомендации в табличный формат (~15Gb memory required)

item_ids_encoded = range(len(item_encoder.classes_))

sim_recs_als = pd.DataFrame({
    "item_id_enc": item_ids_encoded,
    "sim_item_id_enc": sim_item_ids_enc.tolist(), 
    "score": sim_scores_als.tolist()})
sim_recs_als = sim_recs_als \
    .explode(["sim_item_id_enc", "score"], ignore_index=True) \
    .astype({
        'item_id_enc': 'int32',
        'sim_item_id_enc': 'int32',
        'score': 'float32'
    })

# получаем изначальные идентификаторы
sim_recs_als["item_id"] = item_encoder.inverse_transform(sim_recs_als["item_id_enc"])
sim_recs_als["sim_item_id"] = item_encoder.inverse_transform(sim_recs_als["sim_item_id_enc"])

sim_recs_als = sim_recs_als.drop(columns=["item_id_enc", "sim_item_id_enc"])

sim_recs_als

## Сохраним похожие объекты в файл similar.parquet

In [13]:
path = 'data/recsys/'
sim_recs_als = sim_recs_als[["item_id", "sim_item_id", "score"]]
sim_recs_als.to_parquet(path + 'similar.parquet')

Восстановление из файла

In [14]:
import pandas as pd

path = 'data/recsys/'
sim_recs_als = pd.read_parquet(path + 'similar.parquet')

## Проверим визуально как формируются похожие объекты

In [15]:
items = pd.read_parquet(path + "items.parquet")

In [ ]:
track_name = 'We are the champions' 
artist_name = 'Queen' 

target_track = items[
    items['track_name'].str.contains(track_name, case=False) 
    &
    items['artist_name'].str.join(',').str.contains(artist_name, case=False)
    ].head(1)

target_track

In [ ]:
target_item_id = target_track['item_id'].iat[0]
print(f'target for item id = {target_item_id}')
display(target_track)

sim_recs = sim_recs_als[sim_recs_als['item_id'] == target_item_id] \
    .merge(items, left_on='sim_item_id', right_on='item_id')

print('similar items')
display(sim_recs)


Работает - похожие треки выглядят неплохо .. Френк Синатра, Элвис Пресли ..

# Построение признаков

можно выполнить kernel restart

Построим три признака, можно больше, для ранжирующей модели.

Будем рассматривать следующие 5 признаков:
- признаки моделей-генераторов кандидатов
    - оценка (score) трека от персональных als-рекомендаций (коллаборативные рекомендации)
    - оценка похожести трека (score) по неявным жанровым предпочтениям пользователя, выявленным по его истории (контентные рекомендации)
- совместные признаки трека+пользователя
    - индекс популярности трека в общем рейтинге популярности треков
    - индекс популярности жанра в общем рейтинге жанров (1-ый жанр трека)
    - общее количество треков, которые прослушал пользователь

## 1. Оценка (score) трека от персональных als-рекомендаций 
- этот признак уже рассчитан ранее в als_recommendations

## 2. Оценка (score) похожести трека по неявным жанровым предпочтениям пользователя, выявленным по его истории (контентные рекомендации)
- такой признак и модель генерации еще не была ранее создана - реализуем этот механизм в рамках данного этапа
- для определения данного признака подготовим данные для расчета - фактически это будут рекомендации на основе контента (в отличие от коллаборативных рекомендаций ALS-модели)
    - матрицу трек-жанры (item-2-genre) - из каких жанров состоит трек
    - матрицу пользователь-жанры (user-2-genre) - какие жанры предпочитает пользователь по его прослушанным трекам
- для выбранного/предложенного трека вычислим степень похожести через косинусную близость между векторами трек-жанр и пользователь-жанр 
    - забегая вперед - за приемлемое время это сделать было невозможно, поэтому был изучен и использован механизм FAISS (Facebook AI Similarity Search)

Особенности
- в основном (97%) все треки имеют 1 или 2 жанра (см. раздел EDA - Распределение количества жанров по трекам)
    - поэтому похожесть по жанрам будет малоразличимой для разных треков - из-за огромного (сотни тысяч) треков с одинаковыми жанрами
    - это должно снизить эффективность прямого отбора похожих треков в лоб, но как дополнительный признак для ранжирования будет полезен

### Сформируем матрицу item-2-genre
- распределение каждого объекта по всем возможным жанрам
- по ней в дальнейшем будем искать схожие объекты по жанровым предчпочтениям пользователя

In [17]:
import pandas as pd
import joblib

path = 'data/recsys/'
items = pd.read_parquet(path + "items.parquet")

# загрузим кодировщик для объектов
item_encoder = joblib.load(path + 'item_encoder.joblib')

In [ ]:
# идентификаторы жанров не сквозные, поэтому потребуется применить кодирование

from sklearn.preprocessing import LabelEncoder

genre_encoder = LabelEncoder()
genre_encoder.fit(items['genre_id'].explode())

print(f'genres count: {len(genre_encoder.classes_)}')

In [ ]:
# соберем разреженную матрицу item2genre

import numpy as np
from scipy.sparse import csr_matrix

item2genres = items[['item_id', 'genre_id']].explode('genre_id')
item2genres['genre_id_enc'] = genre_encoder.transform(item2genres['genre_id'])
item2genres['item_id_enc'] = item_encoder.transform(item2genres['item_id'])

item2genres_matrix_csr = csr_matrix(
    (np.ones(item2genres.shape[0], dtype=np.int32),
     (item2genres['item_id_enc'], item2genres['genre_id_enc']))
)

item2genres_matrix_csr

### Матрица - все жанры - genres

In [ ]:
import pandas as pd
path = 'data/preprocess/'
genres = pd.read_parquet(path + "genres.parquet")
genres = genres[genres['id'].isin(genre_encoder.classes_)]
genres.shape

### Матрица user-2-genre
- характеризуем жанровые предпочтения пользователя на основе его истории взаимодействия с объектами (треками)

In [1]:
import pandas as pd

path = 'data/recsys/'
events_train = pd.read_parquet(path + "events_train.parquet", columns=['user_id', 'item_id_enc'])

#### user-2-genres - сначала проверим как работает на одном пользователе (выборочном)

In [ ]:
user_id = 100

def get_user2genre_matrix_csr(user_id, events_history, item2genres_matrix_csr):
    user_events = events_history[events_history['user_id'] == user_id][['user_id', 'item_id']]
    user_events['item_id_enc'] = item_encoder.transform(user_events['item_id'])

    user2genres_matrix_csr = item2genres_matrix_csr[user_events['item_id_enc']]

    user2genres_score = user2genres_matrix_csr.mean(axis=0)
    user2genres_score = np.ravel(user2genres_score)
    return user2genres_score.astype('float32')

user2genres_score = get_user2genre_matrix_csr(user_id, events_train, item2genres_matrix_csr)

print(user2genres_score.shape)
user2genres_score

Посмотрим какие жанры предпочтитает пользователь

In [ ]:
def show_user_genres(user2genres_score, genres, show_plot=True):
    user2genres = genres.copy()
    user2genres['score'] = user2genres_score
    user2genres = user2genres[user2genres['score'] > 0].sort_values(by='score', ascending=False).reset_index(drop=True)
    display(user2genres)
    
    if show_plot:
        user2genres.plot.bar(x='name', y='score')

show_user_genres(user2genres_score, genres)

#### top-100 рекомендации для пользователя
- проверим механику формирования топ-100 рекомендаций для одного пользователя
- на основе определения косинусного сходства вектора жанров пользователя и векторов жанров всех объектов

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

top_n = 100

similar_items_scores = cosine_similarity(X=item2genres_matrix_csr, Y=user2genres_score.reshape((1,131))).flatten()
top_n_item_indices = np.argsort(similar_items_scores)[-top_n:][::-1]

items[items['item_id'].isin(item_encoder.inverse_transform(top_n_item_indices))]

#### top-100 рекомендации для всех пользователей 
- теперь выполним генерацию топ-100 жанровых (контентных) рекомендаций для всей базы пользователей

In [ ]:
user_events = events_train[['user_id', 'item_id_enc']].groupby('user_id').agg(item_ids_enc=('item_id_enc', list)).reset_index()
user_events

In [3]:
def get_user2genre_matrix_csr_compact(item2genres_matrix_csr, item_ids_enc):
    user2genres_matrix_csr = item2genres_matrix_csr[item_ids_enc]
    user2genres_score = np.ravel(user2genres_matrix_csr.mean(axis=0)).astype('float32')
    return user2genres_score

In [ ]:
# it will take 7 mins
user2genres_scores = user_events.apply(lambda row: get_user2genre_matrix_csr_compact(item2genres_matrix_csr, row['item_ids_enc']), axis=1)
user2genres_scores

In [ ]:
user2genres_scores_df = pd.DataFrame(user2genres_scores.tolist())
user2genres_scores_df

Сохраним в файл 7 мин работы вычисления матрицы

In [20]:
path = 'data/recsys/'
user2genres_scores_df.to_parquet(path + 'user2genres_scores.parquet')

Восстановим из файла матрицу user2genres_scores
- можно сделать kernel restart (и восстановить вручную item2genres_matrix_csr)

In [ ]:
import pandas as pd

path = 'data/recsys/'
user2genres_scores_df = pd.read_parquet(path + 'user2genres_scores.parquet').rename_axis('user_id')
user2genres_scores_df

In [ ]:
item2genres_matrix_csr

!! Важно:  весь код ниже до Варианта 4/4.1 - является промежуточным/экспериментальным и по сути не пригодился, т.к. Вариант 4.1 потребовал другого подхода

##### Вариант 1 - полное умножение - item2genres (932664, 131) @ user2genres (1372042, 131)
- прямое получение cosine_similarity путем перемножения всей матрицы item2genres (932664 строк) на всю матрицу user2genres (1372042 строк)
- !! не сработал 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# не запускать - будет ошибка
# similar_items_scores = cosine_similarity(X=item2genres_matrix_csr, Y=user2genres_scores_df)

print(similar_items_scores.shape)
display(similar_items_scores)

###### Результат 
- MemoryError: Unable to allocate 9.31 TiB for an array with shape (932664, 1372042) and data type float64
- решение не возможно данным способом

##### Вариант 2 - повекторное item2genres (932664, 131) @ user2genres[i] (1372042, 131)
- перебор в цикле - для каждой строки user2genges получаем cosine_similarity поочерди
- !! не сработал


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# тест через запуск на меньшей партии (head(100))
similar_items_scores = user2genres_scores_df.head(100).apply(lambda row: cosine_similarity(X=item2genres_matrix_csr, Y=np.array(row).reshape(1,-1)), axis=1)

print(similar_items_scores.shape)
display(similar_items_scores)

###### Результат
- примерный расчет необходимого времени - 27 sec / 1000 items - займет более 10 часов
- решение не возможно данным способом 

##### Вариант 3 - прямые матричные операции через numpy
- использование матричных операций напрямую через numpy
- !! не сработал

In [ ]:
import numpy as np

a = item2genres_matrix_csr
b = user2genres_scores_df.head(1) # head(1) для теста

# Не запускать - падение ВМ
'''
a_norm = np.linalg.norm(a, axis=1)
b_norm = np.linalg.norm(b)
similar_items_scores = (a @ b) / (a_norm * b_norm)

print(similar_items_scores.shape)
display(similar_items_scores)
'''

###### Результат 
- падение ВМ по мгновенному перерасходу памяти
    - The Kernel crashed while executing code in the current cell or a previous cell. 
- решение не возможно данным способом

#### Вариант 4 - использование специализированных библиотек
- библиотека Faiss (Facebook AI Similarity Search) - индексирует и квантует вычисления
- использование GPU - Pytorch
- такие объемы и сложности за рамками возможностей обучения ..


##### Вариант 4.1 - Faiss
- pip install faiss-cpu

- при запуске в лоб на подвыборке из 1000 пользователей (user2genres_scores_df) и пачке по 100 штук время работы составило 17,4 сек, что на всей выборке из 1.341 млн записей даст около ~ 6.5ч - неудовлетворительно
- поиск путей ускорения
    - задаем возможность распараллелирования на ядрах CPU - faiss.omp_set_num_threads(4) 
        - время чуть ускорилось до 15.2 сек (вместо 17.4) ~ 5.79 часов
    - заменяем выборку пачки - вместо pandas iloc переходим на прямую numpy индексацию (без копирования в памяти)
    - увеличиваем выборку до 10000 пользователей и размер пачки до 1000
        - получаем - Затраченное время: 75.43 сек, Реальная скорость: 133 пользователей/сек - время работы на всей выборке займет около 3ч
        - все еще неудовлетворительно
- т.к. 3 часа все еще много, перейдем от точного расчета (прямое перемножение матриц - Exact Search - IndexFlatIP) к приближенному - ANNS
    - ANNS - использует иднекс IndexIVFFlat (Inverted File) - деление всех элементов на кластеры и сравнение с ближайшими кластерами - даст точность 98-99%
        - должно ускорить в 100 раз ..?
    - результат
        - время обучения индекса заняло 6 мин 40 сек - что аномально долго - должно быть 10-15 сек
        - время расчета на созданном индексе для 10000 пользователей и 1000 пачки - 
            - Затраченное время: 5.41 сек
            - Реальная скорость: 1849 пользователей/сек        
        - что является отличным ускорением и общее время составит около 12 мин (+ 6 мин обучение)
- ускорим обучение индекса с 6 мин 40 сек до 10-15 сек
    - доработки
        - алгоритму k-means для разметки 4000 кластеров не нужно видеть все 932.644 записей - достаточно случайной выборки из 100-200 тыс объектов - точность при этом не упадет
        - уменьшить количество итераций
            - по-умолчанию FAISS делает 20 итераций k-means, нам для текстовых тегов/жанров (131 признак) достаточно 4-5 итераций
        - убедиться, что обучение работает в режиме многопоточности
    - результат
        - формирование индекса
            - время обучения составило - 39 сек - существенный прогресс
            - время добавления в индекс - 36 сек - тоже ускорилось
            - то есть 75 сек вместо 6 мин 40 сек - отличный результат
        - расчет похожих объектов по всем пользователям, с размером пачки 50000 - 12 мин (users records: 1341269)
            - Расчет завершен, размер матрица индексов - (1341269, 100), дистанций - (1341269, 100)
            - Затраченное время: 732.10 сек
            - Реальная скорость: 1900 пользователей/сек

- результат - 1.3 мин на подгтовку индекса и 12 мин на расчет по всей базе пользователей - отличное время и прогресс

In [ ]:
print(f'item2genres_matrix_csr shape = {item2genres_matrix_csr.shape}')
print(f'user2genres_scores shape = {user2genres_scores_df.shape}')

In [ ]:
import numpy as np
import time
import faiss

start_time = time.time()

# зададим возможность многопоточности - 4 ядра CPU на ВМ
faiss.omp_set_num_threads(4) 

# 1 - подготовка индекса элементов (items)
# приведение к dense-массивам явно требуется для faiss
item2genres_dense = np.ascontiguousarray(item2genres_matrix_csr.toarray().astype('float32'))

# 2 - нормализация векторов (тогда косинусная близость становится матричным произведением) - inplace, directly on memory
faiss.normalize_L2(item2genres_dense)

print(f'done item2genres_dense prepare - {(time.time()-start_time):.0f} sec')

dimensions = item2genres_dense.shape[1] # 131
# 3 - создание индекса на основе скалярного произведение (inner product) - работает долго - не применяем
#index = faiss.IndexFlatIP(dimensions)
#index.add(item2genres_dense)

# 3 - создание индекса IVFFlat - должно существенно ускорить
# задаем параметры кластеризации
# количество корзин
n_list = 4000 # рекомендуется задавать количество центроидов как 4 * sqrt(N) = 4 * sqrt(932000) = 3861
# создаем квантизатор для распределения объектов по корзинам
quantizer = faiss.IndexFlatIP(dimensions)
index = faiss.IndexIVFFlat(quantizer, dimensions, n_list, faiss.METRIC_INNER_PRODUCT)

# ограничиваем количество итераций на кластеризацию до 5 (вместо 20)
index.cp.niter = 5

# ограничиваем выборку для обучения - для 4000 кластеров достаточно 4000 * 39 = 156 тыс строк
n_items_sample = 200000
np.random.seed(19)
rnd_indices = np.random.choice(item2genres_dense.shape[0], size=n_items_sample, replace=False)
items_train_set = np.ascontiguousarray(item2genres_dense[rnd_indices])

# обучение индекса - займет 39 сек
print('start IndexIVFFlat train')
index.train(items_train_set)
print(f'done IndexIVFFlat train - {(time.time()-start_time):.0f} sec')

print('add items to index')

# добавление индекса - займет 36 сек
index.add(item2genres_dense)
print(f'done add index - {(time.time()-start_time):.0f} sec')

# nprobe - сколько соседних кластеров проверять: 1 - быстро, но точность ниже, 64 - оптимально - быстро и 99% точности
index.nprobe = 64

print(f'Индекс объектов создан: добавлено ({index.ntotal}) записей, время - {(time.time()-start_time):.0f} sec')

In [ ]:
# параметры генерации
n_recs = 100
batch_size = 50000 # for memory optimization 

# 4 - запуска расчета по пользователям пачками
# ограничим для тестирования размер пользователей
#users_sample = 100000
users = np.ascontiguousarray(
    user2genres_scores_df
        #.head(users_sample)        
        .to_numpy()
        .astype('float32'))

num_users = users.shape[0]

# списки для сбора результатов
all_distances = []
all_indices = []

print(f'users records: {num_users}')

search_time = time.time()
print(f"Начало рассчет для пользователей... time: {search_time}")

for i in range(0, num_users, batch_size):
    print(f'bacth position: {i}')

    # отбор пачки
    user_batch = users[i:i + batch_size]
    # конвертация в массив
    
    # нормализация
    faiss.normalize_L2(user_batch)

    # поиск топ-n похожих объектов
    # на IndexIVFFlat индексе будут "отсекаться лишние 95% записей"
    distances, indices = index.search(user_batch, n_recs)

    all_distances.append(distances)
    all_indices.append(indices)

    batch_time = time.time() - search_time
    print(f'time spent: {batch_time:.2f} сек')

# 5 - объединение результата

final_distances = np.vstack(all_distances)
final_indices = np.vstack(all_indices)

total_time = time.time() - search_time

print(f'Расчет завершен, размер матрица индексов - {final_indices.shape}, дистанций - {final_distances.shape}')

print(f'Затраченное время: {total_time:.2f} сек')
print(f'Реальная скорость: {(num_users / total_time):.0f} пользователей/сек')

Сохраним сырые расчетные данные в файл
- 13 мин работы генерации

In [11]:
import numpy as np

path = 'data/recsys/'
np.savez_compressed(path + 'user2genre_items_recs_raw.npz', item_ids=final_indices, scores=final_distances)


Восстановим из файла

In [20]:
import numpy as np

path = 'data/recsys/'
loaded_recs = np.load(path + 'user2genre_items_recs_raw.npz')

final_distances = loaded_recs['scores']
final_indices = loaded_recs['item_ids']


#### Преобразуем результат рекомендаций в табличую форму

In [ ]:
import joblib

path = 'data/recsys/'
user_encoder = joblib.load(path + 'user_encoder.joblib')
item_encoder = joblib.load(path + 'item_encoder.joblib')

In [ ]:
import pandas as pd
import numpy as np

# преобразуем полученные рекомендации в табличный формат

missing_items_mask = (final_indices == -1) # там где -1 - значит что FAISS не смог найти похожий объект для данного пользователя
sanitized_indices = final_indices.copy()
sanitized_indices[missing_items_mask] = 0 # временная замена на 0, чтобы item_encoder не ругался на неизвестный id_enc == -1

# получаем изначальные идентификаторы

item_ids = item_encoder.inverse_transform(sanitized_indices.ravel())
item_ids = item_ids.reshape(final_indices.shape).astype(object)
item_ids[missing_items_mask] = None # возвращаем пустые id

user_ids = user_encoder.inverse_transform(user2genres_scores_df.index.to_numpy())

user2genre_items_recs = pd.DataFrame({
    'user_id': user_ids,
    'item_id': list(item_ids), 
    'score': list(final_distances)
    }) 

user2genre_items_recs = user2genre_items_recs.explode(['item_id', 'score'], ignore_index=True)

 # очистка пустых рекомендаций
user2genre_items_recs.dropna(subset=['item_id'], inplace=True)

user2genre_items_recs = user2genre_items_recs.astype({
        'user_id': 'int32',
        'item_id': 'int32',
        'score': 'float32'
    }) \
    .reset_index(drop=True)

print(f"Финальный размер таблицы c рекмоендациями: {user2genre_items_recs.shape}")
# 110.904.153 - то есть 83% от максимальной (1.341.269 пользователя * по 100 топ объектов)
# результат вполне приемлем для приближенного поиска IndexIVFFlat)

user2genre_items_recs

#### Сохраним результат рекомендаций в файл

In [22]:
import pandas as pd

path = 'data/recsys/'
user2genre_items_recs.to_parquet(path + "user2genre_items_recs.parquet")

## Соберем все признаки для обучения ранжирущей модели

тут можно сделать kernel restart

Загрузка данных
- при моделировании и сборке всех признаков потребовалось несколько раз делать промежуточное сохранение в файл 
- и повторную загрузку из файла признаков не на этом этапе, а с момента уже накопленных данных

In [1]:
import pandas as pd

path = 'data/recsys/'

#items = pd.read_parquet(path + "items.parquet")

# 1 - als_recommendations - персональные коллаборативные рекомендации (топ-100 объектов)
# обучено на train-данных - ок
als_recommendations = pd.read_parquet(path + "personal_als.parquet")

# 2 - user2genre_items_recs - рекомендации на основе предпочтений пользователей (контентные) (топ-10 объектов)
user2genre_items_recs = pd.read_parquet(path + "user2genre_items_recs.parquet")

# 3 - tracks_polularity - признаки трека - индекс популярности трека в общем рейтинге популярности треков
# tracks_polularity = ... см. раздел ниже

# 4 - признаки трека - индекс популярности жанра в общем рейтинге жанров (1-ый жанр трека)
# genres_popularity  = ... см. раздел ниже

# 5 - признаки пользователя - общее количество треков, которые прослушал пользователь
# .. см. раздел ниже


#### 1 - персональный als-рекомендации

Подготовка датасета с признаками - features

In [2]:
# 1 - персональный als-рекомендации

features = als_recommendations.rename(columns={'score': 'score_als'}).astype({'score_als': 'float32'})

### 2 - рекомендации на основе жанра пользователей (по контенту)

In [ ]:
# 2 - рекомендации на основе жанра пользователей (по контенту)

features = features.merge(
    user2genre_items_recs.set_index(['user_id', 'item_id']).rename(columns={'score': 'score_genre'}), 
    on=['user_id', 'item_id'], 
    how='outer')
features

In [ ]:
features.info(show_counts=True)

In [ ]:
# чистка памяти
#del als_recommendations
#del user2genre_items_recs

Сохраним в файл промежуточные признаки от als-модели и от близости жанров

In [21]:
import pandas as pd

path = 'data/recsys/'
features.to_parquet(path + "features-1.parquet")


#### Восстановим из файла (точка - 1)
- можно делать kerner restart и подгружать только нужные данные

In [1]:
import pandas as pd

path = 'data/recsys/'
features = pd.read_parquet(path + "features-1.parquet")
events_train = pd.read_parquet(path + "events_train.parquet", columns=['user_id', 'item_id'])

### 3 - общая популярность трека (количество прослушиваний)

In [2]:
# 3 - добавляет признаки трека - общая популярность трека (количество прослушиваний)

tracks_polularity = pd.DataFrame(events_train['item_id'].value_counts().sort_values(ascending=False).astype('int32').reset_index())

features = features.merge(
    tracks_polularity[['item_id', 'count']].set_index('item_id').rename(columns={'count': 'track_popularity_count'}),
    on='item_id',
    how='left'
)
features['track_popularity_count'] = features['track_popularity_count'].astype('int32')

In [ ]:
features

In [ ]:
features.info(show_counts=True)

### 4 - индекс популярности жанра в общем рейтинге жанров (1-ый жанр трека)

In [ ]:
# загрузим данные о треках для связки с genre_id

import pandas as pd

path = 'data/preprocess/'
tracks_flat = pd.read_parquet(path + "tracks_flat.parquet", columns=['track_id', 'genre_id'])

tracks_genre_df = tracks_flat[['track_id', 'genre_id']] \
    .drop_duplicates(subset='track_id') \
    .astype({'genre_id': 'int32'})

tracks_genre_df

In [ ]:
# добавим genre_id для связки
tracks_genre_df = tracks_genre_df.rename(columns={'track_id': 'item_id'})

features = features.merge(
    tracks_genre_df,
    on='item_id',
    how='left'
)

features

In [ ]:
features.info(show_counts=True)

In [9]:
path = 'data/recsys/'
items = pd.read_parquet(path + "items.parquet", columns=['item_id', 'genre_id'])

In [10]:
genres_popularity = pd.DataFrame(events_train['item_id'].value_counts().sort_values(ascending=False).reset_index()) \
    .merge(
        items,
        on='item_id',
        how='left'
    ) \
    .explode(['genre_id'])  \
    .groupby(['genre_id']).agg(count=('count', 'sum')).sort_values(by='count', ascending=False) \
    .reset_index()      

genres_popularity = genres_popularity \
        .rename(columns={'count': 'genre_popularity_count'}) \
        .astype({'genre_popularity_count':'int32'})

Промежуточное сохранение

In [ ]:
path = 'data/recsys/'

features.to_parquet(path + 'features-2.parquet')
genres_popularity.to_parquet(path + 'genres_popularity.parquet')

#### Восстановим из файла (точка - 2)

In [1]:
import pandas as pd

path = 'data/recsys/'

features = pd.read_parquet(path + 'features-2.parquet')
genres_popularity = pd.read_parquet(path + 'genres_popularity.parquet')

In [ ]:
# 4 - признаки трека - индекс популярности жанра в общем рейтинге жанров (1-ый жанр трека)

features = features.merge(
    genres_popularity,
    on='genre_id',
    how='left'
)
features

In [6]:
features.drop(columns='genre_id', inplace=True)

In [ ]:
features.info(show_counts=True)

Сохраним в файл промежуточные признаки после добавление популярности трека и жанра

In [8]:
import pandas as pd

path = 'data/recsys/'
features.to_parquet(path + "features-3.parquet")

#### Восстановим из файла (точка - 3)
- можно делать kerner restart и подгружать только нужные данные

In [1]:
import pandas as pd

path = 'data/recsys/'
features = pd.read_parquet(path + "features-3.parquet")

### 5 - общее количество треков, которые прослушал пользователь

Загрузим данные о событиях для добавления количества прослушенных треков

In [3]:
import pandas as pd

path = 'data/recsys/'
events_train = pd.read_parquet(path + "events_train.parquet", columns=['user_id', 'item_id'])

In [4]:
user_tracks = events_train.groupby(by='user_id').agg(user_track_count=('item_id', 'count')).astype('int16').reset_index()

In [ ]:
# 5 - признаки пользователя - общее количество треков, которые прослушал пользователь

features = features.merge(
    user_tracks,
    on='user_id'
)
features

In [ ]:
features.info(show_counts=True)

In [7]:
# уберем NA-значения в оценках score_als и score_genre, образованных после outer join              
features.fillna(0, inplace=True)

In [ ]:
features.info(show_counts=True)

## Сохраним финальный датасет с признаками

In [9]:
import pandas as pd

path = 'data/recsys/'
features.to_parquet(path + 'features-final.parquet')

# Ранжирование рекомендаций

Построим ранжирующую модель, чтобы сделать рекомендации более точными. Отранжируем рекомендации.

- все пользовательские рекомендации (features датасет), которые были подготовлены, были собраны на обучающем наборе events_train
- ранжирующая модель будет выполнять ранжирование на этих признаках, при этом для ее обучения, необходимо иметь известные целевые значения (таргеты)
    - чтобы получить таргеты, необходимо "заглянуть в будущее", относительно обучающих данных, и выделить объекты, с которыми провзаимодействовал пользователь - эти объекты получат таргет = 1, остальные 0
    - этим будущим является исходная тестовая выборка, которая была изначально отделена от тренировочной - период с 16.12.2022 по 31.12.2022
- для формирования обучающей выборки событий с таргетами для ранжирующей модели разделим исходный тестовый набор на две части 
    - с 16.12 по 23.12.2022 включительно - обучающая выборка с таргетами
    - с 24.12 по 31.12.2022 включительно - тестовая выборка, на которой будем собирать метрики ранжирующей модели

### Загрузим подготовленный набор признаков

In [ ]:
# pandas - не хватает памяти на merge
import pandas as pd

'''
path = 'data/recsys/'
features = pd.read_parquet(path + 'features-final.parquet')
display(features)
display(features.info(show_counts=True))
'''


### Разделение на тестовую и обучающие выборки для ранжирования и установка таргета

Через pandas не получается сделать merge кандидатов и events_test для формирования таргетов - не хватает памяти (19Gb)
- выполним эту операцию через polars
- pip install polars

In [ ]:
# pandas - не хватает памяти на merge
'''
import pandas as pd

path = 'data/recsys/'
# events_test ранее сформирован в начале раздела 3
events_test = pd.read_parquet(path + "events_test.parquet", columns=['user_id', 'item_id', 'started_at'])

# разделяем тестовые и обучающие выборки для ранжирования
train_rank_split = '2022-12-23'
events_rank_train = events_test.query('started_at <= @train_rank_split')
events_rank_test = events_test.query('started_at > @train_rank_split')

print(f'events_rank_train size: {events_rank_train.shape[0]:,} ({(100*events_rank_train.shape[0]/events_test.shape[0]):.2f}%), events_rank_test size: {events_rank_test.shape[0]:,} ({(100*events_rank_test.shape[0]/events_test.shape[0]):.2f}%)')

# установка таргета
events_rank_train['target'] = 1

candidates = features.merge(
    events_rank_train[['user_id', 'item_id', 'target']], 
    on=['user_id', 'item_id'], 
    how='left')

candidates
'''

In [ ]:
# polars - для выполнения в lazy-режиме
import polars as pl
from datetime import datetime

path = 'data/recsys/'
features = pl.scan_parquet(path + 'features-final.parquet')

# выделим обучающую и тестовую выборку для модели ранжирования и зададим целевые значения (таргеты) по взаимодействию
# events_test ранее сформирован в начале раздела 3
events_test = pl.scan_parquet(path + "events_test.parquet")

# разделяем тестовые и обучающие выборки для ранжирования
train_rank_split = datetime(2022, 12, 23)
events_rank_train = events_test.filter(pl.col('started_at') <= train_rank_split)

print(f'events_rank_train size: {events_rank_train.collect().shape[0]:,} ({(100*events_rank_train.collect().shape[0]/events_test.collect().shape[0]):.2f}%)')

# установка таргета
events_rank_train = events_rank_train.with_columns(pl.lit(1).alias('target'))

# соединяем с кандидатами 
candidates = features.join(
    events_rank_train.select(['user_id', 'item_id', 'target']), 
    on=['user_id', 'item_id'], 
    how='left')

# сохраняем промежуточную выборку, чтобы продолжить с pandas
candidates.sink_parquet(path + 'candidates-1.parquet')

### Сформируем сбалансированную выборку для обучения 
- отсечем пользователей, которые не содержат ни одного положительного таргета
- оставим не более 4-х отрицательные кандидатов на пользователя 

Продолжим с pandas после установки таргета с помощью polars

In [ ]:
# pandas
import pandas as pd

path = 'data/recsys/'
candidates_1 = pd.read_parquet(path + 'candidates-1.parquet')

candidates_1

In [ ]:
print(f'target was set for - {candidates_1["target"].notna().sum()} records')

# проставляем target = 0, там где не было взаимодействия
candidates_1['target'] = candidates_1['target'].fillna(0).astype('int8')
candidates_1

In [ ]:
candidates_1.info()

#### Оставляем в выборке кандидатов только тех пользователей, у которых есть хотя бы один положительный таргет

In [ ]:
candidates_to_sample = candidates_1.groupby(by='user_id').filter(lambda x: x['target'].sum() > 1)
candidates_to_sample

In [69]:
def calc_user_targets_info(user_group):
    result = {}
    result['item_count'] = user_group['item_id'].count()
    result['target_1_count'] = user_group['target'].sum()
    result['target_0_count'] = user_group['target'].count() - user_group['target'].sum()
    result['target_1_percent'] = 100 * (user_group['target'].sum() / user_group['target'].count()).round(2)
    return pd.Series(result)

In [70]:
# посмотрим сколько всего таргетов у пользователя и сколько положительных - чтобы избежать сильного дисбаланса

candidates_to_sample_targets_df = candidates_to_sample.groupby(by='user_id').apply(lambda group: calc_user_targets_info(group)) 

# - сейчас там по ~100-200 кандидатов с нулевым таргетом

In [ ]:
candidates_to_sample_targets_df.sort_values(by='target_1_count', ascending=False)

In [ ]:
candidates_to_sample_targets_df.plot.scatter(x='target_1_count', y='target_0_count', xticks=range(0,60,2), yticks=range(0,200, 10), title='Targets per user-items distribution', figsize=(12,6))

#### Для каждого пользователя оставляем пропорцию примерно 25% положительных и 75% негативных примеров

In [121]:
candidates_to_sample_targets_df.reset_index(inplace=True)

In [162]:
def sample_target(group):
    stats_df = candidates_to_sample_targets_df.query(f'user_id == {group.user_id.iat[0]}')
    target_1_count = stats_df['target_1_count'].iat[0]
    target_0_count = stats_df['target_0_count'].iat[0]
    
    sample_0_max = target_1_count * 3
    sample_0_count = int(min(sample_0_max, target_0_count, group.shape[0]))
    return group.sample(sample_0_count)

In [ ]:
candidates_to_sample_trunk = candidates_to_sample.query('target == 0').groupby('user_id').apply(lambda x: sample_target(x))
candidates_to_sample_trunk

In [ ]:
candidates_to_train = pd.concat([
    candidates_to_sample.query('target == 1'),
    candidates_to_sample_trunk
]).reset_index(drop=True).sort_values(by='user_id')

candidates_to_train

In [ ]:
# посмотрим сколько таргетов теперь у пользователя

candidates_to_train_targets_df = candidates_to_train.groupby(by='user_id').apply(lambda group: calc_user_targets_info(group)) 
candidates_to_train_targets_df.sort_values(by='target_1_count', ascending=False)

In [ ]:
candidates_to_train_targets_df.plot.scatter(x='target_1_count', y='target_0_count', xticks=range(0,60,2), title='Targets per user-items distribution', figsize=(12,6))

Видим, что минимальный баланс 1/3 соблюден

#### Сохраним итоговый обучающий набор для модели ранжирования

In [191]:
import pandas as pd

path = 'data/recsys/'
candidates_to_train.to_parquet(path + 'candidates_to_train.parquet')

## Обучение ранжирующей модели
- будем применять catboost классификатор в качестве модели ранжирования
- TODO: для задачи ранжирования также рекомендуется использовать CatBoostRanker, который лучше оптимизирует метрику

In [192]:
import pandas as pd

path = 'data/recsys/'
candidates_to_train = pd.read_parquet(path + 'candidates_to_train.parquet')

In [193]:
candidates_to_train

,user_id,item_id,score_als,score_genre,track_popularity_count,genre_popularity_count,user_track_count,target
281160,10,51447372,0.000000,0.70014,125,51464261,29,0
281159,10,373500,0.000000,0.64312,89,51464261,29,0
281164,10,35515789,0.126154,0.00000,23795,51464261,29,0
123,10,49125069,0.114226,0.00000,42845,51464261,29,1
281162,10,41832275,0.154456,0.00000,28578,14975217,29,0
...,...,...,...,...,...,...,...,...
1124304,1374582,41631554,0.153013,0.00000,15947,23365431,205,0
1124297,1374582,63083897,0.131248,0.00000,11049,2084052,205,0
1124296,1374582,68106316,0.094246,0.00000,12304,17282553,205,0
280998,1374582,79071657,0.160986,0.00000,30252,51464261,205,1


In [197]:
from catboost import CatBoostClassifier, Pool, CatBoostRanker

feature_cols = ['score_als', 'score_genre', 'track_popularity_count', 'genre_popularity_count',	'user_track_count']
X = candidates_to_train[feature_cols]
y = candidates_to_train['target']

n_iterations = 1000
learning_rate = 0.1
depth = 6
loss_function = 'Logloss'
verbose = 1000
random_seed = 91

model_cb = CatBoostClassifier(iterations=n_iterations, learning_rate=learning_rate, depth=depth, loss_function=loss_function, verbose=verbose, random_seed=random_seed)
model_cb.fit(X=X, y=y)

0:	learn: 0.5896041	total: 108ms	remaining: 1m 47s
999:	learn: 0.4014846	total: 1m 38s	remaining: 0us


Сохраним обученную модель в файл

In [199]:
path = 'data/recsys/'
model_cb.save_model(path + 'model_catboost_clsf.cbm')

Восстановление модели из файла

In [5]:
from catboost import CatBoostClassifier

path = 'data/recsys/'
model_cb = CatBoostClassifier().load_model(path + 'model_catboost_clsf.cbm')

## Формирование тестовых рекомендаций

In [1]:
# pandas
import pandas as pd

path = 'data/recsys/'
features = pd.read_parquet(path + 'features-final.parquet')

In [2]:
import pandas as pd
path = 'data/recsys/'

# events_test ранее сформирован в начале раздела 3
events_test = pd.read_parquet(path + "events_test.parquet", columns=['user_id', 'item_id', 'started_at'])

# разделяем тестовые и обучающие выборки для ранжирования
train_rank_split = '2022-12-23'
events_rank_test = events_test.query('started_at > @train_rank_split')

events_rank_test.shape

(5403167, 3)

In [4]:
# выберем кандидаты для ранжирования из тех пользователей, которые есть в тестовом датасете
candidates_to_rank = features[features['user_id'].isin(events_rank_test['user_id'])]
candidates_to_rank

,user_id,item_id,score_als,score_genre,track_popularity_count,genre_popularity_count,user_track_count
800,4,18820599,0.208646,0.0,43918,12267018,234
801,4,694683,0.191520,0.0,62521,2606577,234
802,4,647040,0.186123,0.0,42152,2606577,234
803,4,40493419,0.183279,0.0,29498,515103,234
804,4,986,0.180763,0.0,19272,12267018,234
...,...,...,...,...,...,...,...
242490135,1374582,48591588,0.095507,0.0,26233,23365431,205
242490136,1374582,33308971,0.095246,0.0,24818,743365,205
242490137,1374582,72340123,0.094276,0.0,7757,14975217,205
242490138,1374582,68106316,0.094246,0.0,12304,17282553,205


## Выполнение ранжирование - получение топ-100 предложений

In [7]:
feature_cols = ['score_als', 'score_genre', 'track_popularity_count', 'genre_popularity_count',	'user_track_count']
X = candidates_to_rank[feature_cols]

predictions = model_cb.predict_proba(X)
predictions

array([[0.7938804 , 0.2061196 ],
       [0.93008272, 0.06991728],
       [0.90014304, 0.09985696],
       ...,
       [0.56722339, 0.43277661],
       [0.60056968, 0.39943032],
       [0.6725439 , 0.3274561 ]])

In [8]:
candidates_to_rank['cb_clsf_score'] = predictions[:,1]
candidates_to_rank = candidates_to_rank.sort_values(by=['user_id', 'cb_clsf_score'], ascending=[True, False])
candidates_to_rank['rank'] = candidates_to_rank.groupby(by='user_id').cumcount()+1

candidates_to_rank

/tmp/ipykernel_6065/1229464139.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates_to_rank['cb_clsf_score'] = predictions[:,1]


,user_id,item_id,score_als,score_genre,track_popularity_count,genre_popularity_count,user_track_count,cb_clsf_score,rank
818,4,38646012,0.132286,0.0,29377,29660560,234,0.690686,1
808,4,32851197,0.153471,0.0,35808,51464261,234,0.619599,2
842,4,60292250,0.100394,0.0,74461,3144302,234,0.613671,3
889,4,38903142,0.082427,0.0,15545,17282553,234,0.550437,4
813,4,24327639,0.136376,0.0,18776,2606577,234,0.498621,5
...,...,...,...,...,...,...,...,...,...
242490091,1374582,52395280,0.114465,0.0,39717,17282553,205,0.172292,96
242490136,1374582,33308971,0.095246,0.0,24818,743365,205,0.160815,97
242490068,1374582,9167465,0.140317,0.0,30774,12456152,205,0.149639,98
242490072,1374582,37641523,0.135271,0.0,34741,51464261,205,0.113866,99


Оставим не более топ-100 рекомендаций для пользователя

In [11]:
candidates_to_rank = candidates_to_rank[candidates_to_rank['rank'] <= 100]
candidates_to_rank

,user_id,item_id,score_als,score_genre,track_popularity_count,genre_popularity_count,user_track_count,cb_clsf_score,rank
818,4,38646012,0.132286,0.0,29377,29660560,234,0.690686,1
808,4,32851197,0.153471,0.0,35808,51464261,234,0.619599,2
842,4,60292250,0.100394,0.0,74461,3144302,234,0.613671,3
889,4,38903142,0.082427,0.0,15545,17282553,234,0.550437,4
813,4,24327639,0.136376,0.0,18776,2606577,234,0.498621,5
...,...,...,...,...,...,...,...,...,...
242490091,1374582,52395280,0.114465,0.0,39717,17282553,205,0.172292,96
242490136,1374582,33308971,0.095246,0.0,24818,743365,205,0.160815,97
242490068,1374582,9167465,0.140317,0.0,30774,12456152,205,0.149639,98
242490072,1374582,37641523,0.135271,0.0,34741,51464261,205,0.113866,99


## Сохраним результат ранжирования в файл

In [12]:
path = 'data/recsys/'
candidates_to_rank.to_parquet(path + 'recommendations.parquet')

# Оценка качества

Проверим оценку качества трёх типов рекомендаций: 

- топ популярных,
- персональных, полученных при помощи ALS,
- итоговых
  
по четырем метрикам: recall, precision, coverage, novelty.

### Подготовим матрицу ошибок классификации
- сформируем данные для матрицы ошибок (confusion matrix)
    - TP, FP, TN, FN

In [1]:
import pandas as pd

path = 'data/recsys/'
recommendations = pd.read_parquet(path + 'recommendations.parquet', columns=['user_id', 'item_id', 'cb_clsf_score'])

events_train = pd.read_parquet(path + "events_train.parquet", columns=['user_id', 'item_id', 'started_at'])
events_test = pd.read_parquet(path + "events_test.parquet", columns=['user_id', 'item_id', 'started_at'])

train_rank_split = '2022-12-23'
events_rank_train = events_test.query('started_at <= @train_rank_split')
events_rank_test = events_test.query('started_at > @train_rank_split')

events_train_big = pd.concat([events_train, events_rank_train])

In [2]:
# проверим, что все объекты на которых было выполнено ранжирование были в обучающем наборе - исключим самые свежие объекты при их наличии, так как по ним заведомо не может быть сформировано рекомендаций
events_rank_test = events_rank_test[events_rank_test["item_id"].isin(events_train_big["item_id"])]

In [3]:
def calc_confusine_matrix(events_rank_test, recommendations, top_k = 100):
    
    # gt - "позитивные" объекты - явное наличие объекта в истории взаимодействия
    events_rank_test['gt'] = True

    # top_k
    recs = recommendations.groupby('user_id').head(top_k)

    cm = events_rank_test[['user_id', 'item_id', 'gt']].merge(
        recs[['user_id', 'item_id', 'cb_clsf_score']],
        on=['user_id', 'item_id'],
        how='outer'
    )

    # заполним пустоты, что равнозначно отрицательному значению - т.к. не попал в merge
    cm['gt'] = cm['gt'].fillna(False)

    # pr - наличие рекомендаций по данному объекту
    cm['pr'] = ~cm['cb_clsf_score'].isna()
    # tp - верное попадение
    cm['tp'] = cm['gt'] & cm['pr']
    # fp - неверное попадение
    cm['fp'] = ~cm['gt'] & cm['pr']
    # fn - неверное исключение
    cm['fn'] = cm['gt'] & ~cm['pr']

    return cm


In [4]:
cm = calc_confusine_matrix(events_rank_test, recommendations)
cm

,user_id,item_id,gt,cb_clsf_score,pr,tp,fp,fn
0,4,86836928,True,NaN,False,False,False,True
1,4,87446393,True,NaN,False,False,False,True
2,4,91718971,True,NaN,False,False,False,True
3,4,92133060,True,NaN,False,False,False,True
4,4,94227416,True,NaN,False,False,False,True
...,...,...,...,...,...,...,...,...
58943604,1374582,52395280,False,0.172292,True,False,True,False
58943605,1374582,33308971,False,0.160815,True,False,True,False
58943606,1374582,9167465,False,0.149639,True,False,True,False
58943607,1374582,37641523,False,0.113866,True,False,True,False


## Рассчитаем метрики классификации - precision, recall

In [8]:
def calc_metrix(cm):
    cm_grouper = cm.groupby('user_id')

    # precision = tp/(tp + fp)
    precision = cm_grouper['tp'].sum() / (cm_grouper['tp'].sum() + cm_grouper['fp'].sum())
    precision = precision.fillna(0).mean()

    # recall = tp/(tp + fn)
    recall = cm_grouper['tp'].sum() / (cm_grouper['tp'].sum() + cm_grouper['fn'].sum())
    recall = recall.fillna(0).mean()

    return precision, recall

In [10]:
precision, recall = calc_metrix(cm)

print(f'precision = {100*precision:.2f}%, recall = {100*recall:.2f}%')

precision = 0.23%, recall = 6.65%


Метрики для топ-100: 
- precision = 0.23%
- recall = 6.65%

In [ ]:
# посмотрим на метрики для топ-5 для интереса

top_k = 5
cm = calc_confusine_matrix(events_rank_test, recommendations, 5)
precision, recall = calc_metrix(cm)

print(f'top-{top_k}: precision = {100*precision:.2f}%, recall = {100*recall:.2f}%')

# === Выводы, метрики ===

Основные выводы при работе над расчётом рекомендаций, рассчитанные метрики.